# Penjelasan Arsitektur: Prediksi Kurs USD/IDR dengan Berita (DeBERTa + FinBERT + BERTopic + LSTM)

> Dokumen ini menjelaskan isi `fix-notebook.ipynb` **tanpa perlu membaca kode**: apa alurnya, kenapa tiap keputusan diambil, fitur apa yang dihasilkan, dan apa hasilnya (termasuk apa yang belum berhasil).

---

## 1. Ringkasan Singkat (TL;DR)

| Pertanyaan | Jawaban |
|---|---|
| **Apa yang diprediksi?** | *Log return* harian kurs USD/IDR (dari situ didapat arah **Naik/Turun** dan rekonstruksi nilai kurs). |
| **Datanya apa?** | 1.201 hari perdagangan (Sep 2021 – Sep 2026): kurs jual/beli harian + judul & isi berita yang dikumpulkan per hari. |
| **Idenya apa?** | Gabungkan **pola harga historis** dengan **sinyal dari berita** (makna judul, sentimen finansial, topik berita), lalu diprediksi oleh LSTM kecil. |
| **Model akhir** | Pipeline 2 tahap: **Stage 1** LogReg (prediksi arah dari berita) → **Stage 2** `RegularizedSeqLSTM` (~3.000 parameter, 45 fitur, ensemble 5 seed). |
| **Fokus utama notebook ini** | **Anti-overfitting**: versi sebelumnya Train 62,3% vs Val 48,7%. Kapasitas model dipangkas >90% (58.500 → 3.068 parameter). |
| **Hasil di Test** | BalAcc **52,77%**, DirAcc **59,67%**, RMSE **0,002874** (baseline 0,002888), R² **0,0068**. |
| **Hasil ablation (15 skenario)** | Model 45 fitur **bukan yang terbaik**: ia punya Val BalAcc **terendah** dari 15 skenario, dan membuang grup fitur apa pun justru menaikkan Test BalAcc. Satu-satunya kombinasi yang konsisten membaik di Val dan Test adalah **TS + Sent×Arah** (29 fitur). Lihat Bagian 10. |
| **Kesimpulan jujur** | Overfitting **berhasil ditekan** dan pipeline **bersih dari data leakage**, tetapi **kemampuan prediksinya nyaris sama dengan baseline**. Model hampir selalu menebak "Naik". Lihat Bagian 9. |

---

## 2. Masalah yang Ingin Diselesaikan

Kurs USD/IDR dipengaruhi banyak hal, salah satunya **berita** (konflik geopolitik, harga minyak, suku bunga The Fed, dsb.). Hipotesis penelitian:

> *"Jika berita hari ini diubah menjadi angka (makna teks, sentimen, topik), apakah informasi itu membantu memprediksi pergerakan kurs hari ini?"*

Tantangannya: **data sangat sedikit** (hanya ±811 sampel latih untuk sequence) sedangkan fitur teks sangat banyak dimensi. Kombinasi ini sangat rawan **overfitting** (model menghafal noise, bukan belajar pola). Itulah alasan utama notebook ini diberi label "Anti-Overfitting".

---

## 3. Gambaran Alur Keseluruhan

```mermaid
flowchart TD
    A["1. Data mentah<br/>kurs harian + berita harian"] --> B["2. Split kronologis<br/>70% Train / 15% Val / 15% Test"]
    B --> C["3. Embedding judul<br/>DeBERTa-v3-base (frozen) + PCA-8"]
    B --> D["4. Sentimen finansial<br/>FinBERT (frozen)"]
    B --> E["5. Topik berita<br/>BERTopic + PCA-5 + UMAP-2D"]
    C --> F["6. Stage 1: Logistic Regression<br/>prediksi arah dari berita (23 fitur)<br/>5-Fold OOF di Train"]
    D --> F
    E --> F
    F --> G["7. Fitur interaksi Sentimen x Arah<br/>(15 fitur)"]
    D --> G
    B --> H["8. Fitur time series kausal<br/>(14 fitur)"]
    C --> I["Gabung 45 fitur"]
    D --> I
    E --> I
    G --> I
    H --> I
    I --> J["Bentuk sequence<br/>10 hari x 45 fitur"]
    J --> K["9. Stage 2: RegularizedSeqLSTM<br/>tuning di Validation<br/>+ ensemble 5 seed"]
    K --> L["10. Evaluasi di Test<br/>+ rekonstruksi kurs USD/IDR"]
    K --> M["11. Ablation study<br/>sumbangan tiap grup fitur"]
```

Penjelasan satu kalimat per tahap:

1. **Data & split**: baca data, hitung target, bagi waktu **sebelum** apa pun dipelajari.
2. **DeBERTa**: ubah judul berita menjadi vektor makna, lalu diringkas PCA.
3. **FinBERT**: ukur nada berita (positif/negatif/netral) khusus domain keuangan.
4. **BERTopic**: kelompokkan berita ke dalam tema (perang, minyak, suku bunga, dst.).
5. **Stage 1**: model sederhana yang menebak arah kurs dari berita saja.
6. **Interaksi**: gabungkan keluaran Stage 1 dengan sentimen menjadi sinyal baru.
7. **Time series**: fitur teknikal dari harga masa lalu.
8. **Stage 2**: LSTM kecil membaca 10 hari terakhir dari 45 fitur untuk memprediksi return hari ini.
9. **Evaluasi & ablasi**: ukur di data yang belum pernah dilihat, lalu uji fitur mana yang berguna.

---

## 4. Penjelasan Tahap demi Tahap (Apa → Kenapa → Hasil)

### Tahap 1. Setup & Seed Deterministik

- **Apa**: impor pustaka, set `SEED = 42`, pilih GPU/CPU (`cuda` terdeteksi saat dijalankan).
- **Kenapa**: *seed* tetap membuat hasil **bisa diulang**. Tanpa ini, angka bisa berubah tiap dijalankan dan sulit dipertanggungjawabkan.

### Tahap 2. Pemuatan Data & Split Kronologis di Awal

- **Apa**:
  - `rate_mid = (Kurs Jual + Kurs Beli) / 2`.
  - `log_ret = ln(rate_mid_t / rate_mid_{t-1})` → persentase perubahan harian (dalam bentuk log).
  - Target arah `y_dir = 1` jika `log_ret > 0` (kurs naik / rupiah melemah), selain itu `0`.
  - Judul berita dibersihkan (hapus URL & tag HTML). Berita antar-artikel dalam satu hari dipisah tanda ` | `.
  - Data dibagi **urut waktu**: 70% Train, 15% Val, 15% Test.
- **Kenapa**:
  - **Log return** dipakai karena stasioner (lebih stabil secara statistik daripada level kurs mentah) dan bersifat aditif.
  - **Split kronologis, bukan acak**: pada data waktu, mengacak berarti model "mengintip masa depan". Test harus benar-benar lebih baru daripada Train.
  - **Split dilakukan paling awal**, sebelum scaler/PCA/BERTopic di-fit, supaya **tidak ada kebocoran informasi** dari Val/Test ke proses belajar.
- **Hasil**:

| Split | Jumlah hari | Periode | Proporsi hari "Naik" | Hari ada berita |
|---|---|---|---|---|
| Train | 840 | 2 Sep 2021 – 18 Feb 2025 | 55,4% | 86,1% |
| Val | 180 | 19 Feb 2025 – 21 Nov 2025 | 53,3% | 88,9% |
| Test | 181 | 24 Nov 2025 – 1 Sep 2026 | 57,5% | 97,8% |

> **Catatan penting**: di Test, 57,5% hari adalah "Naik". Artinya **model yang selalu menebak "Naik" sudah otomatis benar ±57,5%**. Ini jadi patokan utama saat menilai hasil (Bagian 9).

### Tahap 3. Embedding Judul: DeBERTa-v3-base (Frozen) + PCA

- **Apa**:
  - Setiap judul unik (3.923 judul) dimasukkan ke `microsoft/deberta-v3-base` → vektor 768 dimensi (rata-rata token berbobot *attention mask*).
  - Vektor per hari = rata-rata vektor semua judul hari itu. Hari tanpa berita = vektor nol.
  - `StandardScaler` + `PCA(8)` **di-fit hanya pada Train**, lalu diterapkan ke Val & Test.
- **Kenapa**:
  - **Frozen** (bobot tidak dilatih ulang): dengan ±800 sampel, fine-tuning model 184 juta parameter pasti menghafal. Cukup pakai pengetahuan bahasa yang sudah dipelajari model.
  - **Judul saja** (bukan isi berita): judul padat, konsisten, dan lebih murah; mengurangi noise dari isi artikel yang panjang.
  - **PCA 768 → 8**: 768 dimensi untuk 800 sampel = *curse of dimensionality*. PCA memampatkan ke komponen terpenting.
  - **PCA di-fit di Train saja**: kalau di-fit di seluruh data, struktur Val/Test "bocor" ke Train.
- **Hasil**: matriks harian `(1201, 768)`; 8 komponen PCA menjelaskan **49,1%** varians. Stage 2 memakai 4 komponen pertama, Stage 1 memakai 8.

### Tahap 4. Sentimen Finansial: FinBERT (Frozen)

- **Apa**: tiap judul dinilai `ProsusAI/finbert` → probabilitas **positif / negatif / netral**, lalu dirata-ratakan per hari. Dari situ dibuat:

| Fitur | Rumus | Arti |
|---|---|---|
| `finbert_pos/neg/neu` | rata-rata probabilitas harian | Komposisi nada berita |
| `kurs_pressure` | `P(neg) − P(pos)` | **> 0** → tekanan kurs naik (rupiah melemah); **< 0** → kurs turun |
| `sent_polarity` | `P(pos) − P(neg)` | Kebalikan dari kurs_pressure |
| `sent_magnitude` | `\|kurs_pressure\|` | Seberapa **kuat** sentimen, apa pun arahnya |
| `sent_ratio` | `P(neg) / (P(pos)+P(neg))` | Dominasi sentimen negatif |

- **Kenapa**:
  - **FinBERT** dilatih pada teks keuangan, jadi lebih paham bahasa pasar daripada model sentimen umum (kata seperti "yield", "hawkish" punya konotasi khusus).
  - **Logika domain**: berita buruk → investor lari ke dolar (aset aman) → USD/IDR naik. Maka sentimen negatif diberi tanda positif pada `kurs_pressure`.
  - Hari tanpa berita diberi `neutral = 1` (netral penuh), bukan dibuang, agar urutan waktu tidak terputus.

### Tahap 5. Topic Modelling: BERTopic + PCA-5 + UMAP-2D

- **Apa**:
  - Setiap berita diubah ke embedding dengan `all-MiniLM-L6-v2`, lalu BERTopic (UMAP → HDBSCAN → c-TF-IDF) menemukan tema. **Di-fit hanya pada 723 hari Train yang berberita.**
  - Hasil: **6 topik** + 1 kelompok "outlier" (−1).
  - Probabilitas topik per hari → PCA-5. Embedding juga diproyeksikan UMAP-2D (koordinat peta tema).
- **Topik yang ditemukan**:

| Topik | Kata kunci dominan | Jumlah hari (Train) |
|---|---|---|
| −1 (outlier) | oil, biden, iran, dollar | 179 |
| 0 | china, market, ceo, cnbc | 288 |
| 1 | ukraine, russia, putin, war | 122 |
| 2 | israel, gaza, hamas | 40 |
| 3 | oil, crude, prices | 39 |
| 4 | treasury, bank, inflation, rates | 35 |
| 5 | ukraine, russia, talks | 20 |

- **Kenapa**:
  - Sentimen hanya menjawab *"nadanya apa"*, topik menjawab *"**tentang apa**"*. Berita perang dan berita suku bunga bisa sama-sama negatif tetapi dampaknya ke kurs berbeda.
  - Fit di Train saja untuk mencegah kebocoran. Val/Test hanya di-`transform`.
  - Topik terlihat **masuk akal secara ekonomi** (perang, minyak, suku bunga, China), ini bukti bahwa ekstraksi tema bekerja walau belum tentu prediktif.

### Tahap 6. Stage 1: Classifier Arah Return (Logistic Regression) + 5-Fold OOF

- **Apa**:
  - `LogisticRegression(C=0.1, class_weight="balanced")` memprediksi **arah (naik/turun) hanya dari berita**.
  - 23 fitur = DeBERTa-PCA (8) + FinBERT & has_news (8) + BERTopic-PCA (5) + UMAP (2).
  - Untuk Train, probabilitas dihasilkan lewat **5-Fold Out-Of-Fold (OOF)**.
- **Kenapa**:
  - **Model linear + regularisasi kuat (`C=0.1`)** sesuai kondisi data kecil dan sinyal lemah.
  - **`class_weight="balanced"`** agar tidak bias ke kelas mayoritas ("Naik").
  - **OOF itu krusial**: kalau Stage 1 memprediksi Train dengan model yang dilatih pada Train yang sama, probabilitasnya terlalu "pintar" (bocor). Stage 2 lalu belajar mempercayai sinyal yang palsu. Dengan OOF, setiap prediksi Train dibuat oleh model yang **belum pernah melihat baris itu**, sehingga distribusinya mirip dengan Val/Test.
  - Keluaran Stage 1 (`pred_dir_prob`) dijadikan **fitur** untuk Stage 2 (*stacking*).
- **Hasil**:

| Split | Acc | BalAcc | ROC-AUC |
|---|---|---|---|
| Train (OOF) | 51,55% | 51,41% | 0,5140 |
| Validation | 53,33% | 53,27% | 0,5009 |
| Test | 55,25% | 53,64% | **0,4803** |

> **Interpretasi**: ROC-AUC ≈ 0,50 berarti **berita saja nyaris tidak bisa memisahkan hari naik vs turun**. Ini temuan penting dan harus disampaikan apa adanya.

### Tahap 7. Fitur Interaksi `Sentimen × Arah`

- **Apa**: menggabungkan **tebakan arah Stage 1** dengan **kekuatan sentimen** menjadi 15 fitur, antara lain:
  - `sent_x_dir = kurs_pressure × pred_dir_signed` (searah atau berlawanan?)
  - `sent_mag_x_dir`, `extreme_amp_x_dir` (versi dikuadratkan, menonjolkan kasus ekstrem)
  - Flag: `flag_naik_sent_tinggi`, `flag_turun_sent_tinggi`, `flag_naik_ekstrem`, `flag_turun_ekstrem`
  - `extreme_naik_pressure`, `extreme_turun_pressure`, `sent_dir_5tier` (skala −2…+2)
- **Kenapa**:
  - Hipotesisnya: berita yang **kuat** dan **searah** dengan tebakan arah lebih bisa dipercaya daripada berita lemah.
  - **Ambang dihitung dari Train saja**: "tinggi" = kuantil 70 (0,4587), "ekstrem" = kuantil 90 (0,8343) dari `sent_magnitude` Train. Tidak memakai data Val/Test.

### Tahap 8. Fitur Time Series Kausal & Penyusunan 45 Fitur

- **Apa**: 14 fitur dari harga masa lalu, **semuanya digeser (`shift`) agar hanya memakai data ≤ t−1**:

| Kelompok | Fitur |
|---|---|
| Return lampau | `ret_lag1`, `ret_lag2`, `ret_lag3`, `ret_lag5` |
| Volatilitas | `vol_5d`, `vol_10d` |
| Jarak ke rata-rata bergerak | `dist_ma5`, `dist_ma10`, `dist_ma20` |
| Momentum | `rsi_14`, `consec_up_days` (hari naik/turun berturut-turut) |
| Biaya transaksi | `spread_lag1` (selisih jual-beli) |
| Kalender | `dow_sin`, `dow_cos` (hari dalam minggu, dikodekan siklik) |

- **Kenapa**:
  - Pola harga (momentum, volatilitas) adalah **baseline klasik** yang harus ada; tanpanya tidak bisa dinilai apakah berita menambah nilai.
  - **Shift ke ≤ t−1** menjamin tidak memakai harga hari ini untuk memprediksi hari ini.
  - **Sin/cos untuk hari**: Jumat dan Senin "berdekatan" secara siklus; angka 0–4 mentah tidak menangkap itu.
- **Penyusunan sequence**: tiap sampel = **10 hari terakhir × 45 fitur** (`seq_len=10`) → target return hari ke-10. Scaler fitur & normalisasi target (`mu_y`, `std_y`) **di-fit di Train saja**. Hasil: `X` berbentuk `(1172, 10, 45)` → Train 811, Val 180, Test 181.

### Katalog Lengkap 45 Fitur Stage 2

| Grup | Jumlah | Isi | Sumber informasi |
|---|---|---|---|
| **TS** | 14 | return lag, volatilitas, MA, RSI, streak, spread, hari | Harga historis |
| **DeBERTa-Title** | 4 | `deberta_pca_0..3` | Makna semantik judul |
| **FinBERT-Sent** | 5 | kurs_pressure, magnitude, pos, neg, neu | Nada berita keuangan |
| **Sent×Arah** | 15 | kurs_pressure, magnitude, has_news, pred_dir_prob/signed, interaksi, flag, 5-tier | Stage 1 + sentimen |
| **BERTopic** | 7 | `topic_pca_0..4`, `topic_umap_0..1` | Tema berita |
| **Total** | **45** | | |

---

## 5. Arsitektur Model: `RegularizedSeqLSTM`

### Alur data di dalam model

```
Input (10 hari × 45 fitur)
   │
   ├─ LayerNorm                     → menstabilkan skala per fitur
   ├─ Gaussian Noise (σ=0.08)       → hanya saat training (augmentasi)
   ├─ Input Dropout (0.25)          → membuang fitur acak
   ├─ Linear 45→16 + Tanh + Dropout → BOTTLENECK (pemampatan fitur)
   │
   ├─ LSTM 1 lapis (hidden 16)  ──► ambil keluaran hari terakhir
   │        │
   │        └─ Head: LayerNorm → Dropout(0.45) → Linear(16→1)
   │
   └─ Skip Connection: Linear(16→1) dari hari terakhir × 0.3
   
Output = Head + 0.3 × Skip  →  prediksi log return (1 angka)
```

### Justifikasi setiap komponen

| Komponen | Fungsi | Alasan |
|---|---|---|
| **Bottleneck 45→16** | Memampatkan 45 fitur ke 16 | Memaksa model belajar representasi ringkas, bukan menghafal tiap fitur |
| **LSTM kecil (hidden 16)** | Membaca urutan 10 hari | Kapasitas kecil = sulit menghafal; cukup untuk pola temporal sederhana |
| **Unidirectional** (konfigurasi terpilih) | Hanya membaca maju | Data waktu memang searah; lebih sedikit parameter dari Bi-LSTM |
| **Gaussian Noise** | Mengganggu input saat latih | Seperti "pelatihan dengan data yang diacak sedikit", model tidak bisa bergantung pada nilai persis |
| **Input Dropout** | Membuang fitur acak | Menghindari ketergantungan pada satu fitur |
| **Head Dropout 0.45** | Meredam keluaran | Regularisasi kuat di lapisan akhir |
| **Skip Connection** | Jalur pintas linear | Memberi model "jalur mudah" sehingga LSTM tidak dipaksa menjelaskan semuanya; mirip regresi linear sebagai fondasi |
| **LayerNorm** | Normalisasi | Stabil pada batch kecil (32) |

**Ukuran model**: konfigurasi terpilih hanya **3.068 parameter**, turun dari ±58.500 pada versi sebelumnya (rasio parameter:sampel dari ±72:1 menjadi ±4:1).

### Pelatihan

| Elemen | Pilihan | Alasan |
|---|---|---|
| **Loss** | `HuberLoss` + 0,05 × *sign hinge* | Huber **tahan terhadap outlier** (return ekstrem tidak mendominasi). Hinge menghukum **salah arah** ringan, supaya model peduli pada arah, bukan hanya besaran |
| **Optimizer** | `AdamW`, `lr=4e-4`, `weight_decay=2e-2` | *Weight decay* kuat menahan bobot membesar |
| **Scheduler** | `CosineAnnealingLR` | Learning rate turun halus, mengurangi osilasi |
| **Gradient clipping** | norm 1,0 | Mencegah ledakan gradien di LSTM |
| **Early stopping** | `min_epochs=10`, `patience=15` | Berhenti saat Val loss berhenti membaik; `min_epochs` mencegah berhenti prematur di epoch 1–3 |
| **Best-weights restore** | Bobot terbaik di Val loss | Memakai model di titik terbaiknya, bukan titik akhir |

---

## 6. Strategi Pemilihan Model: Tuning di Validation + Ensemble 5 Seed

### 6.1 Tuning (6 kandidat konfigurasi)

Enam konfigurasi diuji (variasi ukuran, dropout, noise, weight decay, uni/bi-directional, ada/tidaknya penalti arah). Dipilih lewat **skor anti-overfit** di Validation:

```
Skor = Val BalAcc − 0,35 × max(0, Train BalAcc − Val BalAcc) − 2000 × (Val RMSE − RMSE terkecil)
```

- **Kenapa rumus ini?** Model dengan akurasi Val tinggi tetapi gap Train–Val besar dianggap **tidak dapat dipercaya**, jadi dihukum. RMSE ikut dihitung agar akurasi arah bukan satu-satunya ukuran.
- **Test tidak dipakai sama sekali** dalam pemilihan, hanya Validation.

| Peringkat | cfg | Parameter | Train Bal | Val Bal | Gap | Skor |
|---|---|---|---|---|---|---|
| **1 (terpilih)** | 0 | 3.068 | 53,96 | 50,82 | 3,14 | 49,63 |
| 2 | 3 | 3.224 | 53,13 | 50,52 | 2,61 | 49,53 |
| 3 | 2 | 4.948 | 54,14 | 50,60 | 3,54 | 49,27 |
| 4 | 5 | 3.068 | 53,08 | 49,78 | 3,30 | 48,63 |
| 5 | 4 | 5.292 | 53,16 | 48,07 | 5,09 | 46,18 |
| 6 | 1 | 3.068 | 53,51 | 46,73 | 6,78 | 44,28 |

Perbedaan antar kandidat kecil (selisih beberapa poin pada 180 sampel Val), jadi pemilihan ini **tidak terlalu kuat secara statistik**.

### 6.2 Ensemble 5 Seed (42, 7, 123, 2024, 99)

- **Apa**: konfigurasi terpilih dilatih 5 kali dengan inisialisasi acak berbeda, prediksi **dirata-ratakan**.
- **Kenapa**: satu model kecil sangat sensitif pada inisialisasi. Rata-rata beberapa model **meredam varians**. Terbukti di tabel bawah: Test BalAcc antar seed berkisar **50,0% – 56,4%**.

| Seed | Best Epoch | Train Bal | Val Bal | Gap | Test Bal | Test DirAcc | Test RMSE | Test R² |
|---|---|---|---|---|---|---|---|---|
| 42 | 31 | 55,57 | 48,66 | 6,91 | 50,02 | 56,91 | 0,002895 | −0,0080 |
| 7 | 9 | 53,23 | 51,26 | 1,97 | 56,37 | 61,88 | 0,002885 | −0,0012 |
| 123 | 9 | 57,50 | 51,26 | 6,24 | 54,96 | 59,67 | 0,002862 | 0,0147 |
| 2024 | 12 | 52,32 | 46,43 | 5,89 | 53,95 | 59,67 | 0,002873 | 0,0075 |
| 99 | 24 | 56,75 | 47,32 | 9,43 | 51,71 | 56,91 | 0,002890 | −0,0044 |

---

## 7. Hasil Akhir di Test Set (Data yang Belum Pernah Dilihat)

| Model / Split | RMSE | MAE | R² | DirAcc | BalAcc | F1-Macro |
|---|---|---|---|---|---|---|
| Baseline: tebak return 0 | 0,002902 | 0,002201 | −0,0125 | 42,54% | 50,00% | 29,84% |
| Baseline: rata-rata Train | 0,002888 | 0,002178 | −0,0031 | 57,46% | 50,00% | 36,49% |
| LSTM Ensemble (Train) | 0,003399 | 0,002451 | 0,0321 | 59,56% | 54,79% | 48,85% |
| LSTM Ensemble (Val) | 0,003381 | 0,002496 | 0,0073 | 54,44% | 51,56% | 41,76% |
| **LSTM Ensemble (Test)** | **0,002874** | **0,002155** | **0,0068** | **59,67%** | **52,77%** | **42,94%** |

**Classification report arah (Test):**

| Kelas | Precision | Recall | F1 | Jumlah |
|---|---|---|---|---|
| Turun (0) | 0,8333 | **0,0649** | 0,1205 | 77 |
| Naik (1) | 0,5886 | **0,9904** | 0,7384 | 104 |

**Visualisasi yang dihasilkan notebook** (Bagian 10):
1. Kurva loss Train vs Val per epoch.
2. Confusion matrix arah di Test.
3. Deret waktu log return aktual vs prediksi.
4. Rekonstruksi kurs USD/IDR (`kurs prediksi = kurs kemarin × exp(return prediksi)`) beserta penanda sinyal sentimen ekstrem naik/turun.

---

## 8. Checklist Anti-Data-Leakage (Bisa Dijadikan Poin Kekuatan)

| Risiko kebocoran | Penanganan di notebook |
|---|---|
| Test/Val ikut membentuk model | Split kronologis **sebelum** semua fitur |
| Scaler/PCA/UMAP/BERTopic belajar dari masa depan | Semuanya **di-fit di Train saja** |
| Stage 1 "terlalu pintar" di Train | Prediksi Train lewat **5-Fold OOF** |
| Ambang sentimen memakai data Val/Test | Kuantil 70/90 dihitung dari Train |
| Fitur harga memakai harga hari ini | Semua fitur time series di-**shift ke ≤ t−1** |
| Normalisasi target bocor | `mu_y`, `std_y` dari Train |
| Pemilihan model memakai Test | Tuning & early stopping hanya di **Validation** |

---

## 9. Interpretasi Jujur: Apa yang Berhasil dan Belum

### ✅ Yang berhasil

1. **Overfitting ditekan.** Kapasitas model turun >90%; gap Train–Val BalAcc turun menjadi ±2–9 poin (ensemble: Train 54,8% vs Val 51,6%, selisih ±3 poin), jauh lebih sehat daripada versi sebelumnya (62,3% vs 48,7% = ±13,6 poin).
2. **Pipeline metodologis bersih** (lihat Bagian 8), sehingga hasilnya bisa dipercaya sebagai evaluasi yang jujur.
3. **Ekstraksi fitur teks berjalan baik**: topik yang ditemukan masuk akal (perang, minyak, suku bunga), sentimen finansial terhitung, dan seluruhnya terintegrasi.
4. **Ensemble menstabilkan hasil** dibanding satu seed.

### ⚠️ Yang perlu disampaikan secara terbuka

1. **Kekuatan prediksi sangat lemah.**
   - RMSE Test 0,002874 vs baseline 0,002888 → perbaikan **±0,5%**.
   - R² Test **0,0068**, yaitu praktis nol: model nyaris tidak menjelaskan variasi return.
2. **DirAcc 59,67% menyesatkan bila berdiri sendiri.** Proporsi hari "Naik" di Test adalah 57,5%, dan baseline "selalu Naik" (rata-rata Train) sudah mendapat **57,46%**. Selisih hanya ±2 poin.
3. **Model hampir selalu menebak "Naik".** Recall "Naik" 99,0% sedangkan recall "Turun" hanya **6,5%** (5 dari 77 hari turun terdeteksi). Itu sebabnya **BalAcc (52,77%)** lebih jujur sebagai metrik daripada akurasi.
4. **Berita saja hampir tidak prediktif.** ROC-AUC Stage 1 ≈ 0,50 (Val 0,5009; Test 0,4803).
5. **Selisih antar seed besar** (Test BalAcc 50,0–56,4%), jadi 52,77% sebaiknya dibaca sebagai "sekitar acak", bukan klaim keunggulan.
6. **Best epoch sering di 9–12** (sama dengan batas `min_epochs`), pertanda model tidak menemukan banyak pola yang bisa dipelajari lebih lama.

### 📌 Catatan metodologis tambahan

- **Timing berita**: fitur berita hari *t* dipakai untuk memprediksi return hari *t*. Jika sebagian berita terbit **setelah** kurs hari itu terbentuk, ada risiko *look-ahead* ringan. Perlu dipastikan cut-off waktu berita pada dataset.
- **Stage 1 memakai `StratifiedKFold(shuffle=True)`** pada data berurutan waktu. Untuk Stage 1 yang linear dan hasilnya sekadar fitur ini tidak fatal, tetapi `TimeSeriesSplit` lebih ketat.
- **Kurva loss** pada gambar hanya menampilkan **seed pertama (42)**, bukan seluruh ensemble.
- **Ablation membuktikan 45 fitur terlalu banyak** untuk data sekecil ini (Bagian 10): model Full memiliki Val BalAcc terendah dari seluruh 15 skenario. Ini sejalan dengan temuan di atas bahwa berita saja hampir tidak prediktif.

---

## 10. Ablation Study: Fitur Mana yang Benar-Benar Berguna?

### 10.1 Rancangan

Menggunakan konfigurasi terbaik (`BEST_CFG`) dengan **ensemble 3 seed** (42, 7, 123). Total **15 skenario** dalam 4 jenis:

| Tipe | Contoh | Pertanyaan yang dijawab |
|---|---|---|
| **Full** | 45 fitur | Titik acuan |
| **Drop-one** | Full − BERTopic | Seberapa rugi bila satu grup dibuang? |
| **Only-one** | Hanya FinBERT | Seberapa kuat satu grup sendirian? |
| **TS + one** | TS + DeBERTa | Apakah berita menambah nilai di atas data harga saja? |

> Catatan: baris **Full** di ablation (3 seed) memberi Test BalAcc **51,49%**, sedangkan ensemble 5 seed di Bagian 7 memberi **52,77%**. Selisihnya murni karena jumlah seed berbeda, sekaligus menunjukkan betapa berisiknya angka ini.

### 10.2 Hasil Lengkap

| Skenario | Tipe | #Fitur | Val BalAcc | Test BalAcc (±sd) | Test DirAcc | Test RMSE | Test R² | Δ Val vs Full | Δ Test vs Full |
|---|---|---|---|---|---|---|---|---|---|
| **Full (45 fitur)** | full | 45 | 49,18 | 51,49 (±1,48) | 58,01 | 0,002877 | 0,0045 | 0 | 0 |
| − TS | drop-one | 31 | 50,82 | 52,60 (±0,72) | 59,67 | 0,002892 | −0,0056 | +1,64 | +1,11 |
| − DeBERTa-Title | drop-one | 41 | 52,08 | 57,34 (±1,59) | 62,98 | 0,002871 | 0,0088 | +2,90 | +5,85 |
| − FinBERT-Sent | drop-one | 40 | 50,97 | 54,74 (±0,70) | 60,77 | 0,002886 | −0,0015 | +1,79 | +3,25 |
| − Sent×Arah | drop-one | 30 | 50,97 | 51,83 (±2,21) | 58,01 | 0,002894 | −0,0075 | +1,79 | +0,34 |
| − BERTopic | drop-one | 38 | 53,42 | 55,78 (±1,37) | 60,22 | 0,002881 | 0,0022 | +4,24 | +4,29 |
| Only TS | only-one | 14 | 53,87 | 49,71 (±0,84) | 56,35 | 0,002892 | −0,0055 | +4,69 | −1,78 |
| Only DeBERTa-Title | only-one | 4 | 50,07 | 49,52 (±1,42) | 56,91 | 0,002905 | −0,0147 | +0,89 | −1,97 |
| Only FinBERT-Sent | only-one | 5 | 51,41 | 50,99 (±2,18) | 58,01 | 0,002886 | −0,0015 | +2,23 | −0,50 |
| Only Sent×Arah | only-one | 15 | 51,93 | 52,00 (±1,47) | 58,01 | 0,002895 | −0,0075 | +2,75 | +0,51 |
| Only BERTopic | only-one | 7 | 50,74 | 50,65 (±2,13) | 58,01 | 0,002882 | 0,0010 | +1,56 | −0,84 |
| TS + DeBERTa-Title | TS+one | 18 | 54,69 | 51,88 (±0,66) | 56,91 | 0,002889 | −0,0039 | +5,51 | +0,39 |
| TS + FinBERT-Sent | TS+one | 19 | 53,42 | 48,80 (±3,16) | 54,14 | 0,002898 | −0,0101 | +4,24 | −2,69 |
| **TS + Sent×Arah** | TS+one | 29 | **55,95** | 54,76 (±2,14) | 60,22 | 0,002882 | 0,0009 | **+6,77** | **+3,27** |
| TS + BERTopic | TS+one | 21 | 53,65 | 54,57 (±1,51) | 60,77 | 0,002879 | 0,0031 | +4,47 | +3,08 |

*(Grafik leave-one-group-out di notebook menampilkan kolom Δ Val dan Δ Test dari baris drop-one.)*

### 10.3 Cara Membaca Hasil

**Drop-one: Δ positif berarti membuang grup itu justru *memperbaiki* model.** Jadi grup tersebut lebih banyak menambah noise daripada sinyal.

### 10.4 Temuan Utama

1. **Model 45 fitur bukan model terbaik.** Val BalAcc Full (49,18%) adalah yang **terendah dari 15 skenario**, dan **semua 5 skenario drop-one naik** di Val maupun Test. Dengan ±811 sampel latih, 45 fitur terlalu banyak, sehingga fitur tambahan lebih banyak menyumbang noise.
2. **DeBERTa-Title dan BERTopic paling "mengganggu"**: membuangnya menaikkan Test BalAcc +5,85 dan +4,29, serta Val +2,90 dan +4,24. Keduanya konsisten searah di Val dan Test. Sebagai fitur tunggal, keduanya juga lemah (Only DeBERTa 49,52%; Only BERTopic 50,65% di Test).
3. **Sent×Arah adalah grup berita yang paling bernilai.** Kombinasi **TS + Sent×Arah (29 fitur)** memberi Val BalAcc **tertinggi dari semua skenario (55,95%)** dan Test 54,76%, dengan Δ positif di Val (+6,77) dan Test (+3,27). Ini satu-satunya grup berita yang menambah nilai di atas Only TS pada **kedua** split (Val 53,87→55,95; Test 49,71→54,76).
4. **FinBERT mentah tidak membantu bila ditambahkan ke TS**: TS + FinBERT turun di Test (48,80%, terendah dari semua) dengan simpangan antar seed terbesar (±3,16). Nilainya baru muncul setelah diolah menjadi fitur interaksi (Sent×Arah).
5. **TS saja kuat di Val tetapi gagal di Test**: Only TS 53,87% (Val) vs 49,71% (Test). Pola harga historis tidak stabil lintas periode.
6. **RMSE nyaris tidak berubah di semua skenario**: 0,002871 – 0,002905, dibandingkan baseline rata-rata Train 0,002888. Tidak ada kombinasi fitur yang mengalahkan baseline secara berarti pada ukuran galat.

### 10.5 Peringatan Interpretasi (Penting)

- **Val dan Test sering tidak sependapat.** Contoh: Only TS terbaik di Val di antara only-one tetapi terburuk di Test. Itu tanda kuat bahwa selisih antar skenario banyak yang **berada dalam batas noise**. Dengan ±180 sampel, galat baku BalAcc sekitar ±3,7 poin, sebanding dengan sebagian besar selisih di tabel.
- **Jangan memilih fitur berdasarkan Test.** Skenario "− DeBERTa-Title" tampak terbaik di Test (57,34%), tetapi memilihnya karena angka Test berarti *test-set snooping* dan membuat Test tidak lagi independen. Pemilihan yang sah memakai **Validation**, yang menunjuk **TS + Sent×Arah** (Val tertinggi), lalu Test hanya dipakai sebagai konfirmasi akhir: 54,76%, memang membaik tetapi masih modest.
- **Ablation ini sendiri memakai Test untuk perbandingan**, jadi posisikan sebagai **analisis eksploratif**, bukan klaim akhir.
- Hasil di Test sebaiknya dilaporkan sebagai rentang (mis. 51–55%) dan bukan sebagai satu angka tunggal.

### 10.6 Jawaban atas "Apakah Berita Menambah Nilai?"

| Pertanyaan | Jawaban dari ablation |
|---|---|
| Apakah semua fitur berita diperlukan? | **Tidak.** Membuang DeBERTa dan/atau BERTopic justru memperbaiki hasil. |
| Apakah ada fitur berita yang menambah nilai? | **Ya, terbatas:** fitur interaksi Sent×Arah (TS + Sent×Arah menaikkan Val +2,08 dan Test +5,05 dibandingkan Only TS). |
| Apakah menambah nilai secara meyakinkan? | **Belum.** Selisihnya sebanding dengan noise dan RMSE tidak membaik di atas baseline. |

## 11. Jawaban Singkat untuk Pertanyaan yang Mungkin Muncul

**Q: Kenapa model sekecil itu?**
Data hanya ±811 sampel latih. Model besar menghafal noise (terbukti di versi lama: Train 62,3% vs Val 48,7%).

**Q: Kenapa embedding "frozen", bukan fine-tune?**
Fine-tune ratusan juta parameter pada ±800 sampel pasti overfit. Frozen + PCA mengambil pengetahuan bahasa tanpa melatih ulang.

**Q: Kenapa dua tahap (LogReg lalu LSTM)?**
Stage 1 mengubah fitur teks berdimensi tinggi menjadi satu sinyal arah yang ringkas dan teregularisasi. Stage 2 lalu menggabungkannya dengan pola harga secara sekuensial.

**Q: Kenapa pakai OOF?**
Agar sinyal Stage 1 di Train tidak "terlalu bagus" (bocor), sehingga Stage 2 tidak belajar mempercayai sinyal yang palsu.

**Q: Kenapa BalAcc, bukan akurasi biasa?**
Kelas tidak seimbang (Naik 57,5%). Akurasi bisa tinggi hanya dengan selalu menebak "Naik". BalAcc merata-ratakan recall kedua kelas.

**Q: Apakah model ini layak untuk trading?**
Tidak berdasarkan bukti saat ini. Keunggulan atas baseline terlalu kecil dan tidak stabil antar seed.

**Q: Fitur mana yang paling berguna?**
Berdasarkan ablation: kelompok **Sent×Arah** (interaksi sentimen dan arah Stage 1). Kombinasi TS + Sent×Arah (29 fitur) memberi Val BalAcc tertinggi (55,95%) dan Test 54,76%. DeBERTa dan BERTopic justru cenderung menambah noise.

**Q: Kenapa tidak memakai saja model 29 fitur atau subset terbaik?**
Itu langkah lanjutan yang logis. Syaratnya, subset dipilih lewat **Validation**, bukan Test, lalu dievaluasi ulang di Test sekali saja, agar tidak terjadi *test-set snooping*.

**Q: Apa langkah lanjutan yang masuk akal?**
(1) Ulangi pipeline dengan subset **TS + Sent×Arah** (dipilih lewat Val) dan bandingkan dengan Full pada ensemble 5 seed; (2) tambahkan baseline wajib seperti TS-only LogReg; (3) evaluasi dengan *walk-forward* untuk menguji stabilitas lintas periode; (4) tangani bias "selalu Naik" (mis. kalibrasi ambang keputusan); (5) pastikan cut-off waktu berita; (6) tambahkan variabel makro (BI rate, indeks dolar, harga minyak) yang secara ekonomi lebih langsung memengaruhi kurs.

---

## 12. Kesimpulan

Notebook ini membangun **pipeline end-to-end yang rapi dan bebas kebocoran data**: berita diubah menjadi tiga jenis sinyal (makna, sentimen, topik), digabung dengan fitur harga, lalu diproses oleh LSTM kecil yang dirancang khusus untuk **tidak overfitting**.

- **Rekayasa dan metodologi:** tujuan anti-overfitting **tercapai** (58.500 → 3.068 parameter, gap Train–Val menyempit).
- **Kemampuan prediksi:** hanya **sedikit di atas baseline** (BalAcc 52,77%, R² 0,0068), dan model cenderung menebak "Naik".
- **Ablation:** model 45 fitur ternyata **terlalu besar** (Val BalAcc terendah dari 15 skenario). Dari fitur berita, hanya **Sent×Arah** yang menunjukkan nilai tambah konsisten (TS + Sent×Arah: Val 55,95%, Test 54,76%), sedangkan DeBERTa dan BERTopic cenderung menambah noise.
- **Batas klaim:** selisih antar skenario banyak yang sebanding dengan noise (±3,7 poin pada 180 sampel), sehingga hasil ablation bersifat **eksploratif**, bukan bukti akhir.

Temuan ini konsisten dengan kondisi umum pasar valuta: **sinyal harian dari berita sangat lemah dan sulit dieksploitasi pada data sekecil ini**. Hasil tersebut tetap bernilai ilmiah selama dilaporkan apa adanya.
